# Criteo Uplift Project · 完整学习笔记

学习主线是 **业务问题 → 数据准备 → 分组诊断 → ATE/ITT → 建模 → 验证选型 → Test → 稳定性 → 新人群打分 → 业务规划 → 结论**。

阅读时回答每一步的输入、做什么、输出给谁，不必逐行钻研语法。正式结果来自同一个完整实验。这里读取现成产物，并调用正式指标函数完成一个明确标注的八行教学核验；不重新训练或改写实验结果。

研究问题是在固定触达预算下，怎样识别因广告分配而增加转化的人群。数据只支持公开样本上的离线分析，不等于真实线上广告收入。

In [ ]:
from pathlib import Path
import sys, json, os
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'criteo_uplift').is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from criteo_uplift.config import configure_runtime
configure_runtime()
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, HTML
RUN = Path(os.environ['CRITEO_RESULTS_DIR']).resolve()
manifest = json.loads((RUN / 'manifest.json').read_text())
assert manifest['status'] == 'complete'
print('结果来源：', RUN, '｜建模行数：', manifest['sample_n'])
TABLE_STYLES = [
    {'selector': 'th', 'props': [('background-color', '#edf3f8'), ('padding', '8px')]},
    {'selector': 'td', 'props': [('padding', '8px'), ('border-bottom', '1px solid #e3eaf0')]},
]


## 01 · 数据范围、质量与样本划分

本页读取已保存的 JSON、CSV 和图片，并进行展示专用的格式化、整合与排版。计算、抽样、划分核验与结果写入由 `criteo_uplift` 和主运行流程完成；修改本页的标题、选列或样式不会回写实验产物。

阅读顺序：**分析边界 → 数据质量 → 全量与抽样构成 → 样本划分 → 重叠检查**。表头与类别使用英文，说明和代码注释使用中文。

## 1. 数据集

- **f0–f11**：12 个匿名处理前特征，作为模型输入。
- **treatment**：处理分配；不等于实际广告曝光。
- **conversion / visit**：结果指标；**exposure** 仅用于审计，不加入模型特征，也不用于筛选人群。

公开数据不含可靠用户标识。`row_id` 标识原 CSV 的来源行，不代表用户身份；本页的数量均为行数。当前结果针对公开样本的分配效果，不直接等于线上曝光效果或收益。

In [ ]:
# 只读取审计与抽样说明；本页不加载原始 CSV 或样本缓存。
audit = json.loads((RUN / "data_preparation/audit.json").read_text())
sample_report = json.loads((RUN / "data_preparation/sample_report.json").read_text())
print("分析指标：", manifest["primary_outcome"], "｜抽样随机种子：", sample_report["seed"])
print("抽样方法：", {"all rows": "保留全部来源行", "uniform without replacement, random-priority reservoir": "均匀无放回抽样（随机优先级蓄水池）"}.get(sample_report["sampling"], sample_report["sampling"]))
print("建模特征精度：", sample_report["feature_representation"], "｜原始字段数：", audit["n_columns"])

## 2. 全量数据质量：有没有基础异常？

只展示缺失、非法标签和标签逻辑异常。以下计数直接来自全量审计；缺失与非法标签按单元格计数，逻辑异常按行计数。

In [ ]:
# 每个检查只展示一次；保留原始计数，不在 Notebook 中重做校验。
quality_fields = {
    "missing_or_nonfinite": "Missing / non-finite values",
    "invalid_binary_labels": "Invalid binary-label values",
    "control_exposed": "Control rows with exposure",
    "conversion_without_visit": "Conversions without visit",
}
quality_table = pd.DataFrame([
    {"Check": label, "Count": audit[key]} for key, label in quality_fields.items()
])
display(quality_table.style.hide(axis="index").format({"Count": "{:,.0f}"}).set_table_styles(TABLE_STYLES))


### 完整记录重复指纹：多余行数与分母

重复指纹使用原始精度下的完整记录字段，属于内容重复候选，不能证明同一用户重复出现，也不据此自动删除。全量内部的内容重复与后文的跨集合特征匹配是不同检查。

同一指纹出现 k 次只计 k−1 个多余行，不能把这个数理解成“唯一重复记录种数”。具体计数定义、分母和指纹局限直接来自审计文件。

In [ ]:
# 直接读取审计字段；比例和方法说明由扫描阶段生成。
duplicate_table = pd.DataFrame([
    {"Item": "Full source rows", "Value": f"{audit['n_rows']:,}"},
    {"Item": "Excess fingerprint rows", "Value": f"{audit['duplicate_row_fingerprints']:,}"},
    {"Item": "Excess fraction", "Value": f"{audit['duplicate_row_fraction']:.2%}"},
    {"Item": "Fraction denominator (rows)", "Value": f"{audit['duplicate_fraction_denominator']:,}"},
    {"Item": "Count definition", "Value": audit["duplicate_count_definition"]},
    {"Item": "Fingerprint method / limitations", "Value": audit["duplicate_method"]},
])
display(duplicate_table.style.hide(axis="index").set_table_styles(TABLE_STYLES).set_properties(
    subset=["Value"], **{"text-align": "left", "white-space": "normal", "max-width": "620px", "overflow-wrap": "anywhere"}
))


## 3. 全量与建模样本：分组不均衡与稀疏转化

用一张对照表回答两个问题：处理／对照比例有多不均衡，转化事件有多稀少。`Share` 的分母为所在数据集总行数；`Conversion rate` 的分母为该组行数。全量和建模样本各自的 `Total` 已由计算阶段保存。

主流程按块扫描原始 CSV，统一准备全量审计、建模抽样和诊断统计。建模样本采用随机优先级蓄水池的均匀无放回抽样；C2ST 则独立按组等量抽样，用来比较两组特征分布。建模样本缓存与全量审计缓存分别核验来源和配置。原始统计使用 float64，建模特征缓存使用 float32 以降低内存；两者用途和重叠检查口径需要区分。

In [ ]:
# 仅拼接两份现成汇总；组别标签由数据生成阶段统一。
group_table = pd.concat([
    pd.DataFrame(report["group_summary"]).assign(Dataset=label)
    for label, report in (("Full source", audit), ("Modeling sample", sample_report))
], ignore_index=True)
group_table = group_table[["Dataset", "group", "n", "fraction", "conversion_positive", "conversion_rate"]].rename(columns={
    "group": "Group", "n": "Rows", "fraction": "Share",
    "conversion_positive": "Conversions", "conversion_rate": "Conversion rate",
})
display(group_table.style.hide(axis="index").format({
    "Rows": "{:,.0f}", "Share": "{:.2%}", "Conversions": "{:,.0f}", "Conversion rate": "{:.4%}",
}, na_rep="—").set_table_styles(TABLE_STYLES).set_properties(subset=["Conversion rate"], **{"font-weight": "600", "color": "#175c83"}))

### 建模特征分布

下图读取运行阶段保存的图片，展示各匿名特征的取值分布；纵轴使用对数刻度。本页不重新加载百万行样本或执行直方图分箱。处理组与对照组的 SMD／KS 检查统一放在 本笔记后续章节。

In [ ]:
# 只显示当前实验的图片；缺失时明确提示，不自动计算或借用其他实验的结果。
feature_plot = RUN / "figures/sample_features.png"
if feature_plot.is_file():
    display(Image(filename=str(feature_plot)))
else:
    print("该实验尚未保存特征分布图；本页跳过此图，不触发计算。")

## 4. 样本划分：各集合是否保留足够事件？

主流程按 `treatment × 主结果` 联合分层，划分训练／验证／测试集（60%／20%／20%）。下表只展示各集合、各组的规模和事件数，不重复总体占比与转化率。

训练集用于拟合，验证集用于选择，测试集用于冻结后的独立评价；本页不重新划分数据。

In [ ]:
# 展示已保存的划分汇总；隐藏 DataFrame 默认索引。
split_table = pd.read_csv(RUN / "data_preparation/split_summary.csv")
split_table = split_table.rename(columns={
    "split": "Split", "treatment": "Group", "n": "Rows",
    "conversion_positive": "Conversions", "visit_positive": "Visits",
})
split_table["Split"] = split_table["Split"].replace({"train": "Train", "validation": "Validation", "test": "Test"})
split_table["Group"] = split_table["Group"].replace({0: "Control", 1: "Treatment"})
display(split_table.style.hide(axis="index").format({
    "Rows": "{:,.0f}", "Conversions": "{:,.0f}", "Visits": "{:,.0f}",
}, na_rep="—").set_table_styles(TABLE_STYLES))

## 5. 跨集合核验：来源行与匿名特征分开解释

**来源行核验**检查行号非空唯一、集合间互斥、划分覆盖完整样本。**特征匹配**检查不同集合中相同的匿名特征表示；匹配比例的分母是被检查集合的行数。

当前建模表示为 `float32`，舍入可能增加匹配数量；原始 `float64` 与建模表示的比较属于精度敏感性分析。来源行不同、特征相同，都不能确定是否为同一用户。此页只展示当前结果文件实际记录的检查，不把历史研究数字当作当前实验的新结果。敏感性分析见 本笔记后续章节。

In [ ]:
# 读取已保存的核验结论；旧实验未记录的检查不视为已通过。
overlap = json.loads((RUN / "data_preparation/feature_overlap.json").read_text())
check_labels = {"non_null_unique": "Non-null and unique", "pairwise_disjoint": "Pairwise disjoint", "complete_coverage": "Complete coverage"}
checks = overlap.get("row_id_checks", {})
print("来源行核验：", "；".join(
    f"{label}: " + {True: "Pass", False: "Fail", None: "Not recorded"}[checks.get(key)]
    for key, label in check_labels.items()
))
overlap_rows = [
    {"Checked split": name.capitalize(), "Reference split": "Train", "Feature match rate": fraction}
    for name, fraction in overlap["fraction_matching_training_features"].items()
]
if "fraction_test_matching_validation_features" in overlap:
    overlap_rows.append({"Checked split": "Test", "Reference split": "Validation", "Feature match rate": overlap["fraction_test_matching_validation_features"]})
display(pd.DataFrame(overlap_rows).style.hide(axis="index").format({"Feature match rate": "{:.3%}"}).set_table_styles(TABLE_STYLES))

In [ ]:
# 来源指纹只展示一次，用于确认本页读取的实验来源。
print("来源 SHA256：", manifest["source"]["sha256"])

## 02 · 实验可信度诊断与平均处理效应

本页读取统计产物，可以重新排版表格与绘制图表；单位转换、坐标轴和颜色只用于展示，不重新执行检验、置换、模型训练或结果写入。正文及注释使用中文，表格使用英文标签并隐藏默认索引。

**阅读顺序：平均效果 → 分配比例 → 单变量平衡 → 联合可区分性 → 结果可学习性。** 每个表只回答一个问题，共用的范围、样本量和计算精度在表外说明一次。

## 1. 全量平均效果：差异有多大，区间有多宽？

读取 `evaluation/ate_summary.csv（scope=full_dataset）`。ATE 是处理组结果率减去对照组结果率；表中差值与区间采用百分点（pp），两组结果率及相对提升采用百分比。图表恢复历史双子图设计，两个横轴分别缩放且保留零效应参考；粗区间线、大端点及小空心点帮助辨认较窄的 Visit 区间。

这是全量公开样本的描述性实验背景，不用于模型选择。P 值由同一双侧 pooled z 检验的稳定对数尾概率计算，科学计数文本约保留四位有效数字。`ln(P)` 保存自然对数；普通浮点 P 值下溢为零时，仍可用对数值表达量级，不能把它解释为真实概率为零。

In [ ]:
# 只转换显示单位并格式化，原始结果表保持不变。
ate = pd.read_csv(RUN / "evaluation/ate_summary.csv", dtype={"p_value_text": "string"})
ate = ate.loc[ate.scope == "full_dataset"]
ate_table = pd.DataFrame({
    "Outcome": ate["outcome"].str.capitalize(),
    "Control rate": ate["response_control"], "Treatment rate": ate["response_treatment"],
    "ATE (pp)": ate["ate"] * 100,
    "95% CI (pp)": [f"[{lo * 100:+.4f}, {hi * 100:+.4f}]" for lo, hi in zip(ate["ci_low"], ate["ci_high"])],
    "Relative uplift": ate["relative_uplift"], "P-value": ate["p_value_text"],
    "ln(P)": ate["log_p_value"],
})
display(ate_table.style.hide(axis="index").format({
    "Control rate": "{:.4%}", "Treatment rate": "{:.4%}", "ATE (pp)": "{:+.4f}",
    "Relative uplift": "{:+.2%}", "ln(P)": "{:.3f}",
}, na_rep="—").set_table_styles(TABLE_STYLES).set_properties(subset=["95% CI (pp)"], **{"white-space": "nowrap"}))
# 从当前表格现场绘图；原始列保持比例单位，绘图副本转换为百分点（pp）。
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

ate_plot = (
    ate.set_index('outcome')
    .loc[['conversion', 'visit'], ['ate', 'ci_low', 'ci_high']]
    .mul(100) #转换成pp值
)

with plt.rc_context( # 使用with:设置临时的图片样式参数
    {
        'font.family': 'DejaVu Sans',
        'font.size': 11,
        'axes.spines.top': False,
        'axes.spines.right': False,
        'axes.spines.left': False,
        'axes.edgecolor': '#C8D0D8',
        'axes.labelcolor': '#344454',
        'text.color': '#23313F',
        'figure.facecolor': 'white',
        'axes.facecolor': 'white',
    }
):
    fig, axes = plt.subplots(
        2,
        1,
        figsize=(12, 6.2),
        sharex=False,
        dpi=130,
        constrained_layout=True,
    )
    fig.suptitle(
        'Absolute treatment effects in the full public sample\n'
        '95% confidence intervals | Separate x-axis scales | Dashed line: zero effect',
        fontsize=13,
    )

    for ax, (outcome, row), color in zip(
        axes,
        ate_plot.iterrows(),
        ['#2166AC', '#087F8C'],
    ):
        ax.axvline(
            0,
            color='#536170',
            linestyle='--',
            linewidth=1.3,
            zorder=2,
        )
        ax.errorbar(
            x=[row['ate']],
            y=[0],
            xerr=[
                [row['ate'] - row['ci_low']],
                [row['ci_high'] - row['ate']],
            ],
            fmt='none',
            color=color,
            elinewidth=3.5,
            capsize=10,
            capthick=2.2,
            zorder=3,
        )

        # 单独绘制小型空心点，避免较窄的 Visit 区间被实心标记遮挡。
        ax.plot(row['ate'], 0, marker='o', markersize=4.5,
                markerfacecolor='white', markeredgecolor=color,
                markeredgewidth=1.6, linestyle='none', zorder=4)

        # 每个子图独立缩放，但始终包含零点、点估计和完整 CI；保持线性坐标。
        left = min(0.0, row['ci_low'], row['ate'])
        right = max(0.0, row['ci_high'], row['ate'])
        span = right - left
        padding = 0.08 * span if span > 0 else 0.01
        ax.set_xlim(
            left - padding,
            right + padding,
        )

        ax.set_ylim(-0.45, 0.85)
        ax.set_yticks(
            [0],
            [outcome.capitalize()],
        )
        ax.tick_params(axis='y', length=0, pad=12)
        ax.xaxis.set_major_locator(MaxNLocator(nbins=6))
        
        ax.set_xlabel('Absolute treatment effect (percentage points, pp)')
        ax.set_axisbelow(True)
        ax.grid(axis='x', color='#E6EBEF', linewidth=0.8)

        # 标注使用绘图区相对坐标，放在点和 CI 上方，避免遮挡。
        ax.text(
            0.98,
            0.94,
            f"ATE = {row['ate']:+.4f} pp\n"
            f"95% CI: [{row['ci_low']:.4f}, {row['ci_high']:.4f}] pp",
            transform=ax.transAxes,
            ha='right',
            va='top',
            fontsize=12,
            fontweight='bold',
            color=color,
            linespacing=1.5,
        )

    plt.show()


## 2. 分配比例：公开计数是否接近外部参考？

读取 `srm.json`。85% 是发布数据的外部参考比例，并非已知的原始实验设计参数。这里的显著或不显著均不能直接证明原始线上实验是否通过 SRM 或随机化。

In [ ]:
# 样本量与范围只说明一次，表内聚焦观测比例、参考比例与检验结果。
srm = json.loads((RUN / "diagnostics/srm.json").read_text())
print(f"分析范围：全量公开数据｜Control：{srm['n_control']:,} 行｜Treatment：{srm['n_treatment']:,} 行")
srm_table = pd.DataFrame([srm])[["observed_ratio", "expected_ratio", "chi2", "pvalue", "status"]].rename(columns={
    "observed_ratio": "Observed treatment share", "expected_ratio": "Reference share",
    "chi2": "Chi-square", "pvalue": "P-value", "status": "Status",
})
display(srm_table.style.hide(axis="index").format({
    "Observed treatment share": "{:.6%}", "Reference share": "{:.2%}", "Chi-square": "{:.3e}", "P-value": "{:.3e}",
}, na_rep="—").set_table_styles(TABLE_STYLES))
print("参考资料：", srm["reference_source"])

## 3. 单变量平衡：哪些特征存在分布差异？

读取 `balance.csv`；范围与上节相同，使用原始 CSV 的 `float64` 表示。样本量、范围、精度和指纹不再逐特征重复列出。

SMD 用两组均值差除以组内方差的合并尺度，表示标准化后的平均差；KS D 比较两组经验分布函数的最大距离，能反映均值以外的分布差异。

表格按已保存的 `|SMD|` 排序，只保留两组均值、带方向的 SMD、KS 的 **D 统计量**及辅助 P 值。D 是经验分布函数的最大距离，不是乘以根号样本量后的量。大样本下优先看差异大小；KS P 值未经多重校正，不能仅凭显著性判定实际重要性。

In [ ]:
# 选取与特征直接相关的列；排序只改变展示顺序，不重新计算统计量。
balance = pd.read_csv(RUN / "diagnostics/balance.csv")
balance_table = balance.sort_values("abs_smd", ascending=False)[[
    "feature", "mean_control", "mean_treatment", "smd", "ks_statistic", "ks_pvalue",
]].rename(columns={
    "feature": "Feature", "mean_control": "Control mean", "mean_treatment": "Treatment mean",
    "smd": "SMD", "ks_statistic": "KS D", "ks_pvalue": "KS P-value",
})
display(balance_table.style.hide(axis="index").format({
    "Control mean": "{:.5f}", "Treatment mean": "{:.5f}", "SMD": "{:+.6f}", "KS D": "{:.6f}", "KS P-value": "{:.3e}",
}, na_rep="—").set_table_styles(TABLE_STYLES).set_properties(subset=["SMD", "KS D"], **{"font-weight": "600", "color": "#175c83"}))

## 4. 联合可区分性：模型能否区分两组？

C2ST 从全量来源按组等概率无放回抽样，构造两组等量的诊断样本；它不等于建模样本，也不使用正式测试集选模。

固定分类器为标准化 Logistic 与 XGBoost（30 棵树、最多 15 个叶节点，直方图训练）。`binary:logistic` 是树模型内部优化目标，外部检验统计量仍为留出 ROC-AUC；辅助保留 Log loss、Brier 和 AP。内部训练／留出各自置换 treatment，并完整重拟合固定分类器。关注留出 AUC 与 Holm 校正后的 P 值：AUC 的随机基准为 0.5；不显著不证明联合分布相同，显著也不能单独证明随机化失败。

In [ ]:
# 共用的抽样与置换配置只在表外展示；表内每行只保留模型自身结果。
c2st = pd.read_csv(RUN / "diagnostics/c2st.csv")
details = json.loads((RUN / "diagnostics/c2st_details.json").read_text())
print(f"诊断样本：Control {details['n_control']:,} 行 + Treatment {details['n_treatment']:,} 行；内部训练／留出为 70%／30%。")
print(f"随机种子：{details['seed']}｜完整重拟合置换：{details['permutations']:,} 次｜P 值分辨率：{details['pvalue_resolution']:.4f}｜两模型 Holm 校正。")
c2st_table = c2st[["model", "roc_auc", "pvalue", "pvalue_holm", "log_loss", "brier", "average_precision"]].rename(columns={
    "model": "Model", "roc_auc": "Holdout ROC AUC", "pvalue": "Permutation P-value", "pvalue_holm": "Holm P-value",
    "log_loss": "Log loss", "brier": "Brier", "average_precision": "AP",
})
# 常数预测基线：前三项指标已保存；常数评分的 AP 等于已保存的留出集正例比例。
# 基线只用于比较预测表现，不新增置换检验或参与两模型的 Holm 校正。
baseline = c2st.iloc[0]
baseline_table = pd.DataFrame([{
    "Model": "dummy",
    "Holdout ROC AUC": baseline["dummy_roc_auc"],
    "Permutation P-value": float("nan"),
    "Holm P-value": float("nan"),
    "Log loss": baseline["dummy_log_loss"],
    "Brier": baseline["dummy_brier"],
    "AP": baseline["test_treatment_ratio"],
}])
c2st_table = pd.concat([baseline_table, c2st_table], ignore_index=True)
print("Dummy 基线对所有行输出内部训练集的 Treatment 占比；这里只展示预测指标，不进行额外检验。")
display(c2st_table.style.hide(axis="index").format({
    "Holdout ROC AUC": "{:.4f}", "Permutation P-value": "{:.3f}", "Holm P-value": "{:.3f}", "Log loss": "{:.5f}", "Brier": "{:.5f}", "AP": "{:.4f}",
}, na_rep="—").set_table_styles(TABLE_STYLES))
# 直接从原来的 c2st_null.csv 现场绘制置换零分布。
null = pd.read_csv(
    (RUN / 'diagnostics' / 'c2st_null.csv')
)
import matplotlib.pyplot as plt
fig, axes = plt.subplots(
    1, 2, figsize=(11, 4), constrained_layout=True,
)
for ax, row in zip(axes, c2st.itertuples()):
    ax.hist(
        null.loc[null['model'] == row.model, 'roc_auc'],
        bins=24, color='#2171ad', alpha=.75, edgecolor='white',
    )
    ax.axvline(
        row.roc_auc, color='#ba6b24', label=f'Observed {row.roc_auc:.4f}',
    )
    ax.set_title(f'{row.model} | Holm p = {row.pvalue_holm:.3f}')
    ax.set_xlabel('Diagnostic holdout ROC AUC')
    ax.set_ylabel('Permutation count')
    ax.legend(frameon=False)
fig.suptitle(
    f"Full-source balanced C2ST | {details['n_control']:,} control + "
    f"{details['n_treatment']:,} treatment\n"
    f"{details['permutations']} complete-refit permutations; internal 70/30 split"
)
plt.show()

## 5. 结果可学习性：特征能预测结果吗？

读取 `learnability.csv`，默认查看整个训练／验证集范围（`whole`），按结果指标分别比较 Dummy 与固定模型。可修改下方 `LEARNING_SCOPE` 以查看某个已保存的分组，选项以输出提示为准。

Dummy 只输出训练集正例率，作为没有特征信息的基准；模型在训练集拟合、在验证集评价。ROC AUC／AP 越高越好，Log loss／Brier 越低越好；AP 应结合验证集正例率阅读。结果可预测不等于 CATE 可预测，也不能代替策略增量评价。

In [ ]:
# 仅筛选已有范围与结果；各模型共用的样本量和正例率在表外展示。
LEARNING_SCOPE = "whole"
learning = pd.read_csv(RUN / "diagnostics/learnability.csv")
print("已保存的分析范围：", ", ".join(learning["scope"].unique()))
learning_view = learning.loc[learning["scope"] == LEARNING_SCOPE]
if learning_view.empty:
    print("该结果文件没有所选范围，请从上面的范围中选择。")
for outcome in learning_view["outcome"].unique():
    rows = learning_view.loc[learning_view["outcome"] == outcome]
    context = rows.iloc[0]
    print(f"分析指标：{outcome}｜范围：{LEARNING_SCOPE}｜训练 {context['n_train']:,} 行，验证 {context['n_validation']:,} 行")
    print(f"正例率：训练 {context['positive_rate_train']:.4%}，验证 {context['positive_rate_validation']:.4%}")
    table = rows[["model", "roc_auc", "average_precision", "log_loss", "brier", "status"]].rename(columns={
        "model": "Model", "roc_auc": "ROC AUC", "average_precision": "AP", "log_loss": "Log loss", "brier": "Brier", "status": "Status",
    })
    display(table.style.hide(axis="index").format({
        "ROC AUC": "{:.4f}", "AP": "{:.4f}", "Log loss": "{:.5f}", "Brier": "{:.5f}",
    }, na_rep="—").set_table_styles(TABLE_STYLES))

本页只展示保存的诊断，不在展示时重跑检验。新的 Overlap 诊断由主流程在四个候选训练前统一执行；完整实验请使用 run_pipeline.py 并指定新的输出目录，保留已冻结结果。详见 [项目流程](../docs/PROJECT_PIPELINE.md)。

## S / T / X：算法与基模型分开理解

S-Learner 拟合一个 $\hat\mu(X,T)$；T-Learner 分别拟合 $\hat\mu_0(X)$、$\hat\mu_1(X)$。两者的 CATE 都是 $\hat\mu_1(X)-\hat\mu_0(X)$。

X-Learner 先用两个结果模型构造伪效应，再拟合连续效应回归器：

$$D_0=\hat\mu_1(X_0)-Y_0,\qquad D_1=Y_1-\hat\mu_0(X_1)$$
$$\hat\tau(X)=e\hat\tau_0(X)+(1-e)\hat\tau_1(X)$$

这里 $e$ 沿用训练集 Treatment 占比。X-Learner 返回第一阶段的 `mu0`、`mu1` 和第二阶段加权后的 `cate`。

`models.s_learner()`、`t_learner()`、`x_learner()` 各自实现完整的拟合和预测。提供训练数据时拟合一次并预测，传入 `fitted` 时直接预测。正式候选仍是 S＋Logistic、T＋Logistic、T＋XGBoost、X＋XGBoost。树数默认 150，其他树参数保持原配置；Validation 只选候选策略，当前项目没有做早停或超参数搜索。Test 始终不参与选择。

## Overlap / Positivity：训练集经验覆盖

新流程在 S/T/X 拟合前，只对 Train 运行一次逐特征分位数分层诊断；四个候选共用结果。默认最多 10 箱、每箱任一臂不足 20 人时警告，阈值可配置，缺失值独立成箱。单变量两臂覆盖不能证明高维联合正值性；局部缺臂也可能来自有限样本。

预期分配概率必须说明来源，不能把约 0.85 或样本比例自动当成已经核实的设计概率。一致性、无干扰和可交换性仍需实验设计与业务机制支持。结果标签单类别属于分类器拟合边界，另在训练摘要中记录，不是因果不可识别的结论。

此单元格读取当前实验已保存的训练集分箱结果。


In [ ]:
overlap_path = RUN / 'diagnostics/overlap_positivity.json'
coverage = json.loads(overlap_path.read_text())
display({key: coverage[key] for key in ('scope', 'status', 'n', 'n_treatment', 'n_control',
                                       'observed_treatment_rate', 'probability_source',
                                       'missing_arm_bins', 'weak_bins', 'design_positivity')})
display(pd.read_csv(RUN / 'diagnostics/overlap_strata.csv'))
print(coverage['interpretation'])


## 03 · 候选 uplift 模型与 validation 选型

## 本页要回答什么

项目有哪些候选模型？模型分数如何进入策略排序？最终策略和 uplift 候选是如何在 validation 上冻结的？

## 代码导航

- 模型定义与训练函数：criteo_uplift/models.py 的 s_learner()、t_learner()、x_learner()。
- 训练与冻结选型：criteo_uplift/training_validation.py 的 fit_and_select_models()。
- 排序指标：criteo_uplift/metrics.py 的 evaluate_ranking()。
- 结果文件：validation_metrics.csv、validation_bootstrap.csv、selection.json、模型 .joblib 文件、calibration_*.csv。

本页先看候选模型与验证集证据，再看冻结选择。通用 S/T/X learner 原理请查 DSDAHANDBOOK；本页只解释本仓库的实现与结果。

## 候选模型与实现边界

当前候选为 S-Logistic、T-Logistic、T-XGBoost、X-XGBoost；同时比较随机排序与 response propensity 基线。模型训练与预测函数集中在 models.py，训练、验证和冻结选择集中在 training_validation.py。

阅读顺序：先看 models.s_learner()、t_learner()、x_learner() 的算法步骤，再看 training_validation.fit_and_select_models() 如何只用 train 拟合、只用 validation 选择。response propensity 是强基线，不等于 treatment assignment probability。

In [ ]:
# 只展示模型类型、训练耗时和常见配置，避免把所有 sklearn 参数铺成一张超宽表。
model_rows = []
for name, info in manifest['models'].items():
    params = info.get('parameters', {}).get('outcome_params', {})
    model_rows.append({
        'model': name,
        'fit_seconds': info.get('seconds'),
        'n_estimators': params.get('n_estimators'),
        'max_depth': params.get('max_depth'),
        'learning_rate': params.get('learning_rate'),
        'C': params.get('C'),
    })
model_table = pd.DataFrame(model_rows).set_index('model')
display(model_table.style.format({'fit_seconds': '{:.2f}'}))

validation = pd.read_csv(RUN / 'models/validation_bootstrap.csv')
display(validation.query("policy != 'random'").sort_values('bootstrap_mean', ascending=False, kind='stable'))


## 两项整体指标与三个预算 Gain

本页读取 Validation 和 Test 的指标表。AUUC 积分使用 Uplift Gain，传统 Qini Score 扣除对应参考线面积。覆盖率决定预算 Gain 的 Top-k，整体面积使用完整排序前缀。


In [ ]:
ranking_columns = ['policy', 'auuc', 'qini_score']
for label, filename in [
    ('Validation ranking', 'models/validation_metrics.csv'),
    ('Frozen test ranking', 'evaluation/test_metrics.csv'),
]:
    print(label)
    ranking = pd.read_csv(RUN / filename)
    display(ranking[ranking_columns].drop_duplicates('policy').reset_index(drop=True))
display(Image(filename=str(RUN / 'figures/ranking.png')))


## CATE 校准诊断

**实现位置：** evaluation.py 的 calibration_table()；测试阶段由 evaluation.py 保存 calibration_*.csv。

分箱结果用于检查预测分数与组间观测差异是否呈现一致趋势。它不是个体真实反事实标签，因此不能据此报告真实 PEHE/ITE MSE；全局均值接近 ATE 也不能证明个体排序正确。测试集诊断不会用于重新校准当前模型。

In [ ]:
chosen_uplift = manifest['selection']['selected_uplift_model']
print('Validation 冻结的最佳 uplift 候选（不一定是最终策略）:', chosen_uplift)
calibration = pd.read_csv((RUN / 'evaluation' / f'calibration_{chosen_uplift}.csv'))
display(calibration.head(12))
display(Image(filename=str(RUN / 'figures/calibration.png')))

## 训练与预测分离

模型算法位于 models.py，Train 拟合与 Validation 选型位于 training_validation.py。Test 仅调用已拟合模型，所有预处理也只在 Train 拟合。Overlap 诊断独立保存，不传入 Learner。

~~~python
from criteo_uplift.models import x_learner
validation_result = x_learner(validation[FEATURES], X_train=train[FEATURES],
                              t=train.treatment, y=train.conversion,
                              outcome_params={'random_state': 42, 'n_estimators': 150},
                              effect_params={'random_state': 42, 'n_estimators': 150})
tau_test = x_learner(test[FEATURES], fitted=validation_result['model'])['cate']
~~~

本 Notebook 只读取公开聚合结果，以上是接口说明，不在 Notebook 内执行训练。
完整流程和文件说明见 ../docs/PROJECT_PIPELINE.md 与 ../docs/RESULTS_GUIDE.md。


## 04 · 冻结策略的离线评价

Test 复用 Validation 冻结的策略与主指标。完整排序指标和多预算 Gain 从 test_metrics.csv 读取，只有主指标的推断写入 test_bootstrap.csv 和 test_pairwise.csv。计算入口是 metrics.py 的 evaluate_ranking()。


## 先冻结策略，再评价 Test

Validation 在五个评分策略中选择主指标 Bootstrap 均值最高的策略，并写入 selection.json。Random 用于参考比较。Test 展示冻结策略的独立评价。

HT Gain 的每十万人标度以全部符合条件的待选用户为分母。面积指标覆盖完整排序曲线。


In [ ]:
selection = json.loads((RUN / 'models/selection.json').read_text())
print('冻结策略', selection['selected_policy'], '主指标', selection['primary_metric'])
metrics = pd.read_csv(RUN / 'evaluation/test_metrics.csv')
test_boot = pd.read_csv(RUN / 'evaluation/test_bootstrap.csv')
test_pairs = pd.read_csv(RUN / 'evaluation/test_pairwise.csv')
display(test_boot)


## 成对 bootstrap：区间应怎样阅读

**实现位置：** metrics.py 的 evaluate_ranking()；测试阶段由 evaluation.py 调用。

同一次按 treatment 臂分层的重抽样用于比较策略，每次按固定分数重新选 top-k。策略差值使用成对差值分布；不要用两条独立置信区间是否重叠来判断策略差异。

当前区间条件于已拟合模型、冻结分数、固定的分配概率估计和观测臂样本量；不包含重训、模型选型、分配概率估计误差或未知用户依赖。多个辅助预算比较属于探索性分析。

In [ ]:
display(Image(filename=str(RUN / 'figures/policy.png')))
budget_columns = ['policy', 'budget', 'n_selected', 'gain_plugin', 'qini_traditional', 'gain_ht_per_100k']
display(metrics[budget_columns])


## 解读冻结策略的区间

选型依据是 Validation 主指标的 Bootstrap 均值。Test 的差值区间来自同一轮配对抽样，各预算使用点态区间。获胜频率描述固定评分下的选择稳定性。投放成本和价值情景见 Planning。


In [ ]:
selected = selection['selected_policy']
print('冻结策略', selected)
display(test_pairs[test_pairs.policy.eq(selected) & test_pairs.reference.isin(['random', 'propensity'])])
print('完整报告', RUN / 'report.html')


## 下一步阅读

- 想知道 test 指标怎样计算：criteo_uplift/evaluation.py。
- 想知道测试集评价如何调用指标并写出 CSV/NPZ：criteo_uplift/evaluation.py。
- 想知道报告图表如何生成：criteo_uplift/reporting.py。

主流程与结果目录导航见 ../docs/PROJECT_PIPELINE.md；各结果文件含义见 ../docs/RESULTS_GUIDE.md。

## 指标教学核验 · 八行人工数据

以下仅验证公式实现，不是 Criteo 业务结果。所有正式指标由 metrics.py 计算。

In [ ]:
from criteo_uplift.metrics import evaluate_ranking
t = np.array([1,0,1,0,1,0,1,0])
y = np.array([1,0,1,0,0,1,1,0])
scores = np.arange(8,0,-1)/10
result = evaluate_ranking(y, t, {'example': scores}, e=.5, n_bootstrap=0)
curve, metrics = result['curves'], result['metrics']
display(pd.DataFrame({'rank':np.arange(1,9),'score':scores,'treatment':t,'conversion':y}))
display(curve[['n_selected', 'fraction', 'n_treatment', 'n_control', 'uplift_rate', 'gain_plugin', 'qini_traditional', 'gain_ht']])

## 项目函数计算结果

直接展示 metrics.py 返回的指标，再用同一份曲线结果绘图。Notebook 不重复实现积分或统计检验；精确分数基准与公式正确性由 tests/test_evaluation.py 验证。

相同排序使用不同的曲线定义时，面积数值可能不同，不能混用指标口径。


In [ ]:
display(metrics[['auuc', 'qini_score']].drop_duplicates())
# 仅选择用于绘图的节点。
plot_curve = curve[['fraction', 'gain_plugin', 'qini_traditional']].dropna()
q = plot_curve.fraction.tolist()
g = plot_curve.gain_plugin.tolist()
qini = plot_curve.qini_traditional.tolist()


In [ ]:
fig,axes=plt.subplots(2,2,figsize=(12,8),layout='constrained')
axes[0,0].plot(curve.fraction,curve.uplift_rate,'o-',label='L = rate difference')
axes[0,0].axhline(.5,ls='--',color='gray',label='Overall L(1)')
axes[0,0].set(title='Cumulative lift',ylabel='Incremental rate among selected')
axes[0,1].plot(curve.fraction, curve.gain_ht, 'o-', color='#d47a20', label='HT gain')
axes[0,1].plot(curve.fraction, curve.random_gain_ht, '--', color='gray', label='Endpoint reference')
axes[0,1].set(title='HT gain', ylabel='Incremental responses / eligible person')
axes[1,0].plot(list(map(float,q)),list(map(float,g)),'o-',label='Plug-in gain')
axes[1,0].fill_between(list(map(float,q)),list(map(float,g)),alpha=.15)
axes[1,0].plot([0,1],[0,.5],'--',color='gray',label='Endpoint reference')
axes[1,0].set(title='Uplift curve: AUUC = 81/256',ylabel='Incremental responses / eligible person')
axes[1,1].plot(list(map(float,q)),list(map(float,qini)),'o-',color='#167f67',label='Traditional Qini')
axes[1,1].plot([0,1],[0,.25],'--',color='gray',label='Endpoint reference')
axes[1,1].fill_between(list(map(float,q)),list(map(float,qini)),[float(x/4) for x in q],alpha=.15,color='#167f67')
axes[1,1].set(title='Qini curve: centered score = 19/384',ylabel='Qini count / total sample')
for ax in axes.flat:
    ax.set_xlabel('Selected population fraction');ax.legend(fontsize=9);ax.grid(alpha=.2)

plt.show()

## HT 与 plug-in 曲线为什么分别保留

本项目同时保存 HT 与传统 plug-in 估计口径，原因是有限样本下两者的前缀估计方式不同。这里通过同一组人工数据并排画图，检查代码实现是否按预定定义计算；它不是用来判断哪个模型在真实 Criteo 数据上更好。

实现位置仍是 metrics.py 的 evaluate_ranking()。

In [ ]:
fig,ax=plt.subplots(figsize=(9,4),layout='constrained')
ax.plot(curve.fraction,curve.gain_ht,'o-',label='HT gain: AUUC=3/8')
ax.plot(list(map(float,q)),list(map(float,g)),'s--',label='Plug-in gain: AUUC=81/256')
ax.plot([0,1],[0,.5],':',color='gray',label='Endpoint reference')
ax.set(xlabel='Selected population fraction',ylabel='Gain per eligible person',title='Same ranking, different finite-sample estimators')
ax.legend();ax.grid(alpha=.2)
plt.show()

## Stability · 排除 Test 后检查稳定性

采用原设计的种子 17/29/83、嵌套训练比例 25%/50%/100%，并比较行分层划分与匿名特征分组划分。每个配置的 Validation 只用于本次开发实验。

这些实验共享开发样本，均值、标准差与选择频率是描述性统计，不是新的独立验证。相同匿名特征不是用户标识；之前已经看过历史 Test，排除其行不能消除分析者已有知识。

In [ ]:
ST = RUN / 'stability'
report = json.loads((ST / 'report.json').read_text())
long = pd.read_csv(ST / 'longmetrics.csv')
agg = pd.read_csv(ST / 'aggregate.csv')
audit = pd.read_csv(ST / 'split_audit.csv')

display({
    key: report[key]
    for key in [
        'experiments', 'fitted_models', 'development_n',
        'old_test_n_excluded', 'all_old_test_row_overlaps_zero',
        'all_group_split_feature_overlaps_zero'
    ]
})
audit_columns = [
    'split_method', 'train_n', 'validation_n',
    'train_old_test_overlap', 'validation_old_test_overlap',
    'feature_group_overlap'
]
missing_columns = [column for column in audit_columns if column not in audit.columns]
if missing_columns:
    raise ValueError(f"split_audit.csv 缺少必需字段：{', '.join(missing_columns)}")
if 'row_overlap' in audit.columns:
    audit_columns.append('row_overlap')
display(audit[audit_columns])

cols = [
    'split_method', 'training_fraction', 'policy',
    'gain_ht_per_100k_mean', 'gain_ht_per_100k_std',
    'vs_propensity_per_100k_mean', 'train_control_events_min',
    'train_control_events_max'
]
display(agg[cols].sort_values(['split_method', 'training_fraction', 'policy']))

## 读取开发期与独立 Test 的结果

上方的 Stability 汇总描述不同训练规模和划分下的开发结果。独立 Test 主指标与基线的配对区间见本 Notebook 的冻结策略评价一节。


## Planning · 明确假设之后做规划

`plan_policy_ab()` 由假设的线上基线率、MDE、分配比、alpha 和 power 推算两组样本量。`cost_value_scenario()` 将离线增量按假设的每次转化价值和触达成本换算经济价值。

这里沿用原项目的教学参数。所有经济数字都依赖固定价值、固定成本和效果可迁移假设，不能说成 Criteo 已验证的真实收入；固定假设换算的区间也不包含成本或价值的不确定性。

In [ ]:
assumptions = json.loads((RUN / 'business/assumptions.json').read_text())
display(assumptions)
display(pd.read_csv(RUN / 'business/hypothetical_ab_plans.csv'))
display(pd.read_csv(RUN / 'business/hypothetical_cost_scenarios.csv'))

## 项目收束与面试表达

1. **业务目标**：固定触达预算下增加转化，区分响应概率与增量效果。
2. **可信流程**：全量检查和审计、均匀抽样、60/20/20 分层划分、分组诊断、Train 拟合、Validation 冻结、Test 评价。
3. **方法证据**：ATE/ITT 说明平均分配效果；CATE 排序用于定向；HT 策略值与配对 Bootstrap 描述预算收益及差异的不确定性。
4. **稳健性**：排除 Test 后检查随机种子、训练规模、特征分组划分；不把开发期赢家当成新独立证据。
5. **业务落地边界**：独立打分接口已经具备；真实投放仍需要新数据、明确成本与新的线上 A/B。

简历只写真实完成的方法、规模与结果。相对 response 的区间包含零应表述为证据不足；不能写“提升了真实收入”。最终策略始终以 Validation 冻结记录为准。

In [ ]:
selection = manifest['selection']
summary = pd.read_csv(RUN / 'evaluation/test_bootstrap.csv')
chosen = summary[summary.policy.eq(selection['selected_policy'])]
print('冻结策略', selection['selected_policy'], '最佳 uplift 候选', selection['selected_uplift_model'])
display(chosen)
print('完整报告', RUN / 'report.html')
